# DeepGuard AI - Notebook 03: EfficientNet-B0 Model Training
### Project: Deepfake Image & Video Detection System
**Objective:** Fine-tune ImageNet-pretrained EfficientNet-B0 on extracted facial crops for binary classification (REAL vs FAKE).
Implements Automatic Mixed Precision (AMP), AdamW, Cosine Annealing, Early Stopping, and Model Checkpoint Export.

### Step 1: GPU Verification & Libraries

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

!pip install timm scikit-learn matplotlib
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import json
import os
import time
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using computing device:", device)

---
### Step 1B: Direct URL Dataset Downloader (Zero Manual Upload)
Optionally download the deepfake dataset directly in Colab without downloading to your PC or uploading to Google Drive.


In [ ]:
# Clone repository if needed to load downloader module
import os
if not os.path.exists('Deep-Learning'):
    !git clone https://github.com/AliKhan-Devs/Deep-Learning.git
%cd Deep-Learning/07_DeepGuard_AI/training
import sys
sys.path.append(os.path.abspath('src'))

from datasets.downloader import download_openrl_deepfake, build_manifest_from_folders
DATA_DIR = '/content/data'
print('Downloading OpenRL/DeepFakeFace directly via URL...')
real_dir, fake_dir = download_openrl_deepfake(DATA_DIR)
manifest_df = build_manifest_from_folders(real_dir, fake_dir, os.path.join(DATA_DIR, 'manifest.csv'), max_per_class=3000)
print('>>> Dataset ready for training:', len(manifest_df), 'samples')


### Step 2: EfficientNet-B0 Deepfake Architecture
We load pretrained weights and replace the classifier head with a dropout-regularized linear layer for 2 classes (REAL, FAKE).

In [ ]:
class EfficientNetB0Detector(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, dropout=0.3):
        super().__init__()
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.backbone = efficientnet_b0(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = EfficientNetB0Detector().to(device)
print(f"EfficientNet-B0 initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")

### Step 3: Dataset & Transforms
Training includes RandomHorizontalFlip, ColorJitter, and ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ManifestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

### Step 4: Training Loop with AMP & AdamW

In [ ]:
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

print("Training setup ready. Attach your DataLoader to start training.")

### Step 5: Export Trained Checkpoint
Save `best_model.pth` and companion `model_config.json` to Google Drive so you can easily download them to your local project directory:
`07_DeepGuard_AI/backend/artifacts/models/efficientnet_b0/`

In [ ]:
EXPORT_DIR = "/content/drive/MyDrive/DeepGuard_Models/efficientnet_b0"
os.makedirs(EXPORT_DIR, exist_ok=True)

model_config = {
    "model_name": "efficientnet_b0",
    "display_name": "EfficientNet-B0 Deepfake Detector",
    "architecture": "efficientnet_b0",
    "num_classes": 2,
    "classes": ["REAL", "FAKE"],
    "positive_class": "FAKE",
    "version": "1.0.0",
    "input_size": [224, 224],
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
    "decision_threshold": 0.50,
    "preprocessing": {"face_crop_margin": 0.25, "interpolation": "bilinear", "color_space": "RGB"}
}

# Save config
with open(os.path.join(EXPORT_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

print(f"Model config saved to {EXPORT_DIR}/model_config.json")
# When training completes, save state dict:
# torch.save({"state_dict": model.state_dict(), "config": model_config}, os.path.join(EXPORT_DIR, "best_model.pth"))
# from google.colab import files
# files.download(os.path.join(EXPORT_DIR, "best_model.pth"))
